In [2]:
!pip install -q mediapipe

import os, urllib.request
import mediapipe as mp
print("mediapipe:", mp.__version__)

os.makedirs("/kaggle/working/models", exist_ok=True)
urls = {
    "pose_landmarker_lite.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "pose_landmarker.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
}
for name, url in urls.items():
    dst = f"/kaggle/working/models/{name}"
    if not os.path.exists(dst):
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as e:
            print("فشل", name, e); continue
    print(name, "✅", f"{os.path.getsize(dst)/1e6:.1f} MB")

mediapipe: 1.2.0
pose_landmarker_lite.task ✅ 5.8 MB
pose_landmarker.task ✅ 9.4 MB
hand_landmarker.task ✅ 7.8 MB


In [3]:
# الاستخراج في الخلفية
import os, glob, shutil, subprocess, pandas as pd

shutil.rmtree("/kaggle/working/keypoints", ignore_errors=True)
subprocess.run("pkill -f extract_worker.py", shell=True)

vd = glob.glob("/kaggle/input/**/ASL_Citizen", recursive=True)
assert vd, "لم أجد داتا سيت ASL Citizen، أضفها عبر Add Input"
BASE = vd[0]
print("المسار:", BASE)

POSE_MODEL = "pose_landmarker_lite.task" if os.path.exists("/kaggle/working/models/pose_landmarker_lite.task") else "pose_landmarker.task"
N_RANDOM, N_WORKERS, STRIDE, MAX_MIN = 40, 4, 2, 180

df = pd.concat([pd.read_csv(f"{BASE}/splits/{s}.csv").assign(split=s) for s in ["train", "val", "test"]])
chosen = pd.Series(df["Gloss"].unique()).sample(N_RANDOM, random_state=0).tolist()
sub = df[df["Gloss"].isin(chosen)].sample(frac=1, random_state=0)
sub.to_csv("/kaggle/working/manifest.csv", index=False)
print(f"glosses: {len(chosen)} | فيديوهات: {len(sub)} | نموذج pose: {POSE_MODEL}")

script = r'''
import os, sys, time, cv2, numpy as np, pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

shard, n_shards, max_min, stride, pose_model, base = int(sys.argv[1]), int(sys.argv[2]), float(sys.argv[3]), int(sys.argv[4]), sys.argv[5], sys.argv[6]
VID_DIR = f"{base}/videos"
MODELS = "/kaggle/working/models"
OUT = "/kaggle/working/keypoints"
os.makedirs(OUT, exist_ok=True)
KEEP_POSE = list(range(17))

pose_lm = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=f"{MODELS}/{pose_model}"),
    running_mode=vision.RunningMode.VIDEO, num_poses=1))
hand_lm = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=f"{MODELS}/hand_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO, num_hands=2))
ts_base = 0

def to_arr(lms, n):
    return np.array([[p.x, p.y, p.z] for p in lms[:n]], dtype=np.float32)

def assign_hands(hr, pose):
    left = np.full((21, 3), np.nan, np.float32)
    right = np.full((21, 3), np.nan, np.float32)
    hands = [to_arr(h, 21) for h in hr.hand_landmarks]
    if not hands:
        return left, right
    lw, rw = pose[15, :2], pose[16, :2]
    if np.isnan(lw[0]) or np.isnan(rw[0]):
        for h, hd in zip(hands, hr.handedness):
            if hd[0].category_name == "Left": left = h
            else: right = h
        return left, right
    d = lambda h, w: np.linalg.norm(h[0, :2] - w)
    if len(hands) == 1:
        h = hands[0]
        if d(h, lw) <= d(h, rw): left = h
        else: right = h
    else:
        h1, h2 = hands[:2]
        if d(h1, lw) + d(h2, rw) <= d(h1, rw) + d(h2, lw): left, right = h1, h2
        else: left, right = h2, h1
    return left, right

def extract(path):
    global ts_base
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or np.isnan(fps) or fps <= 0: fps = 30.0
    frames, i = [], 0
    while True:
        if i % stride != 0:
            if not cap.grab(): break
            i += 1
            continue
        ok, img = cap.read()
        if not ok: break
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ts = ts_base + int(i * 1000 / fps)
        pr = pose_lm.detect_for_video(mp_img, ts)
        hr = hand_lm.detect_for_video(mp_img, ts)
        pose = np.full((25, 3), np.nan, np.float32)
        if pr.pose_landmarks: pose = to_arr(pr.pose_landmarks[0], 25)
        left, right = assign_hands(hr, pose)
        frames.append(np.concatenate([pose[KEEP_POSE], left, right]))
        i += 1
    cap.release()
    ts_base += int(i * 1000 / fps) + 1000
    return np.stack(frames)

def normalize(seq):
    c = (seq[:, 11, :] + seq[:, 12, :]) / 2
    s = np.nanmedian(np.linalg.norm(seq[:, 11, :2] - seq[:, 12, :2], axis=1))
    if not np.isfinite(s) or s < 1e-6: return None
    return (seq - c[:, None, :]) / s

files = pd.read_csv("/kaggle/working/manifest.csv")["Video file"].tolist()[shard::n_shards]
t0, done = time.time(), 0
for vf in files:
    if (time.time() - t0) / 60 > max_min:
        print("time budget reached"); break
    out = f"{OUT}/{os.path.splitext(vf)[0]}.npy"
    if os.path.exists(out): continue
    try:
        seq = normalize(extract(f"{VID_DIR}/{vf}"))
        if seq is None: raise ValueError("bad shoulders")
        np.save(out, seq.astype(np.float32))
        done += 1
    except Exception as e:
        with open(f"/kaggle/working/fails_{shard}.txt", "a") as f:
            f.write(f"{vf}\t{e}\n")
print("finished", done)
'''
open("/kaggle/working/extract_worker.py", "w").write(script)

procs = [subprocess.Popen(["python", "/kaggle/working/extract_worker.py", str(i), str(N_WORKERS),
                           str(MAX_MIN), str(STRIDE), POSE_MODEL, BASE],
                          stdout=open(f"/kaggle/working/log_{i}.txt", "w"), stderr=subprocess.STDOUT)
         for i in range(N_WORKERS)]
print("بدأت المعالجة ✅")

المسار: /kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen
glosses: 40 | فيديوهات: 1225 | نموذج pose: pose_landmarker_lite.task
بدأت المعالجة ✅


In [5]:
import glob, time
total = len(pd.read_csv("/kaggle/working/manifest.csv"))
n1 = len(glob.glob("/kaggle/working/keypoints/*.npy"))
time.sleep(60)
n2 = len(glob.glob("/kaggle/working/keypoints/*.npy"))
print(f"تم: {n2}/{total} ({n2/total:.1%}) | السرعة: {n2-n1} فيديو/دقيقة | عمليات حية: {sum(p.poll() is None for p in procs)}/{len(procs)}")
!cat /kaggle/working/fails_*.txt 2>/dev/null | head

تم: 1225/1225 (100.0%) | السرعة: 0 فيديو/دقيقة | عمليات حية: 0/4


In [6]:
import shutil, os
shutil.make_archive("/kaggle/working/keypoints_backup", "zip", "/kaggle/working", "keypoints")
shutil.copy("/kaggle/working/manifest.csv", "/kaggle/working/manifest_backup.csv")
print(f"{os.path.getsize('/kaggle/working/keypoints_backup.zip')/1e6:.0f} MB")

17 MB


In [7]:
import os, glob, cv2, time, warnings, numpy as np, pandas as pd
from scipy.signal import savgol_filter
warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    from numba import njit, prange
except ImportError:
    raise SystemExit("شغّل  !pip install numba  ثم أعد تشغيل الخلية")

W = "/kaggle/working"
SRC = f"{W}/keypoints"
man = pd.read_csv(f"{W}/manifest.csv")
man["stem"] = man["Video file"].str.replace(".mp4", "", regex=False)
info = man.set_index("stem")

vd = glob.glob("/kaggle/input/**/ASL_Citizen/videos", recursive=True)
VID_DIR = vd[0] if vd else None
print("مجلد الفيديوهات:", VID_DIR)

MIN_T, PAD, MAX_GAP = 15, 4, 5
LH, RH = 17, 38
WRIST = {LH: 15, RH: 16}

# ---------- 1) تحميل + تصحيح نسبة الأبعاد ----------
def get_aspect(stem):
    if VID_DIR is None: return None
    cap = cv2.VideoCapture(f"{VID_DIR}/{stem}.mp4")
    w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    return (w / h) if h > 0 else None

raw, asp_missing = {}, 0
for f in sorted(glob.glob(f"{SRC}/*.npy")):
    stem = os.path.basename(f)[:-4]
    r = get_aspect(stem)
    if r is None: r, asp_missing = 4 / 3, asp_missing + 1
    x = np.load(f).copy()
    x[..., 0] *= r; x[..., 2] *= r
    x /= np.nanmedian(np.linalg.norm(x[:, 11, :2] - x[:, 12, :2], axis=1))
    raw[stem] = x
print(f"تم تحميل {len(raw)} | بدون أبعاد (افترضنا 4:3): {asp_missing}")

# ---------- 2) قرار العكس لكل مصوّر (بحسب حركة الرسغين) ----------
def path_len(xy): return np.nansum(np.linalg.norm(np.diff(xy, axis=0), axis=1))
mo = pd.DataFrame([dict(stem=s, pl=path_len(x[:, 15, :2]), pr=path_len(x[:, 16, :2])) for s, x in raw.items()])
mo["pid"] = mo.stem.map(info["Participant ID"])
mo["lr"] = np.log((mo.pl + 1e-3) / (mo.pr + 1e-3))

T_ONE, MIN_N, MIN_SHARE = 0.7, 4, 0.85
g = mo.groupby("pid").agg(n=("stem", "size"),
                          left_only=("lr", lambda v: (v > T_ONE).sum()),
                          right_only=("lr", lambda v: (v < -T_ONE).sum()))
g["tot"] = g.left_only + g.right_only
g["left_share"] = (g.left_only / g.tot.replace(0, np.nan)).round(2)
def decide(r):
    if r.tot < MIN_N: return "unknown"
    if r.left_share >= MIN_SHARE: return "flip"
    if r.left_share <= 1 - MIN_SHARE: return "right"
    return "mixed"
g["decision"] = g.apply(decide, axis=1)
g.to_csv(f"{W}/participant_decisions.csv")
flip_set = set(g.index[g.decision == "flip"])
print("قرارات:", g.decision.value_counts().to_dict(), "| سيُعكس:", sorted(flip_set), "(القديم: P17 P49 P6 P9)")

SWAP = [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]
def mirror(s):
    m = s.copy(); m[..., 0] *= -1
    for a, b in SWAP: m[:, [a, b]] = m[:, [b, a]]
    l, r = m[:, 17:38].copy(), m[:, 38:59].copy()
    m[:, 17:38], m[:, 38:59] = r, l
    return m

n_flip = 0
for s in raw:
    if info.loc[s, "Participant ID"] in flip_set:
        raw[s] = mirror(raw[s]); n_flip += 1
print("فيديوهات معكوسة:", n_flip)

os.makedirs(f"{W}/keypoints_canon", exist_ok=True)
for s, x in raw.items():
    np.save(f"{W}/keypoints_canon/{s}.npy", x.astype(np.float32))

# ---------- 3) قالب اليد من train فقط ----------
def hand_template(base, stems):
    rel = []
    for s in stems:
        h = raw[s][:, base:base + 21]
        ok = ~np.isnan(h[:, 0, 0])
        if ok.sum() < 4: continue
        y = h[ok, 0, 1]
        sel = h[ok][y >= np.percentile(y, 75)]
        rel.append(sel - sel[:, :1])
    return np.median(np.concatenate(rel), axis=0).astype(np.float32)

train_stems = [s for s in raw if info.loc[s, "split"] == "train"]
T_L, T_R = hand_template(LH, train_stems), hand_template(RH, train_stems)
print(f"train: {len(train_stems)} | طول اليد في القالب: {np.linalg.norm(T_L[12]-T_L[0]):.2f} / {np.linalg.norm(T_R[12]-T_R[0]):.2f}")

# ---------- 4) أدوات التنظيف ----------
def interp_nan(v):
    n = np.isnan(v)
    if n.any() and not n.all():
        t = np.arange(len(v)); v = v.copy(); v[n] = np.interp(t[n], t[~n], v[~n])
    return v

def merge_short_gaps(hand, v):
    hand, v2 = hand.copy(), v.copy()
    T, i = len(v), 0
    while i < T:
        if v[i]: i += 1; continue
        j = i
        while j < T and not v[j]: j += 1
        if i > 0 and j < T and (j - i) <= MAX_GAP:
            for k in range(i, j):
                w = (k - (i - 1)) / (j - (i - 1))
                hand[k] = (1 - w) * hand[i - 1] + w * hand[j]
            v2[i:j] = True
        i = j
    return hand, v2

def smooth_runs(hand, v2, win, order):
    out, T, i = hand.copy(), len(hand), 0
    while i < T:
        if not v2[i]: i += 1; continue
        j = i
        while j < T and v2[j]: j += 1
        n = j - i
        w = min(win, n if n % 2 == 1 else n - 1)
        if w >= 5 and w >= order + 2:
            out[i:j] = savgol_filter(hand[i:j], w, order, axis=0, mode="interp")
        i = j
    return out

def fill_hand(hand, v2, pw, tmpl, B):
    out = hand.copy()
    idx = np.where(v2)[0]
    base = pw[:, None, :] + tmpl[None]
    for k in np.where(~v2)[0]:
        left, right = idx[idx < k], idx[idx > k]
        val = base[k].copy()
        for ref in (left[-1] if len(left) else None, right[0] if len(right) else None):
            if ref is None: continue
            w = 1.0 if B <= 0 else min(1.0, abs(k - ref) / B)
            off = hand[ref, 0] - pw[ref]
            shp = hand[ref] - hand[ref, 0]
            val += (1 - w) * (off[None] + shp - tmpl)
        out[k] = val
    return out

def clean2(x, win, order, B):
    T0 = len(x)
    v = np.stack([~np.isnan(x[:, LH, 0]), ~np.isnan(x[:, RH, 0])], axis=1)
    if T0 < MIN_T or not v.any(): return None
    if np.isnan(x[:, :17]).all(axis=0).any(): return None
    idx = np.where(v.any(1))[0]
    a, b = max(0, idx[0] - PAD), min(T0, idx[-1] + PAD + 1)
    x, v = x[a:b].copy(), v[a:b]
    T = len(x)
    for j in range(17):
        for c in range(3):
            x[:, j, c] = interp_nan(x[:, j, c])
    w = min(win, T if T % 2 == 1 else T - 1)
    if w >= 5 and w >= order + 2:
        x[:, :17] = savgol_filter(x[:, :17], w, order, axis=0, mode="interp")
    for h, base, tmpl in [(0, LH, T_L), (1, RH, T_R)]:
        hand, vv = merge_short_gaps(x[:, base:base + 21], v[:, h])
        hand = smooth_runs(hand, vv, win, order)
        x[:, base:base + 21] = fill_hand(hand, vv, x[:, WRIST[base]], tmpl, B)
    return x

# ---------- 5) DTW ----------
@njit
def dtw1(a, b):
    n, m = a.shape[0], b.shape[0]
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = 0.0
            for k in range(a.shape[1]):
                d = a[i - 1, k] - b[j - 1, k]
                s += d * d
            D[i, j] = np.sqrt(s) + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return D[n, m] / (n + m)

@njit(parallel=True)
def dtw_matrix(A, la, B, lb):
    out = np.empty((A.shape[0], B.shape[0]))
    for i in prange(A.shape[0]):
        for j in range(B.shape[0]):
            out[i, j] = dtw1(A[i, :la[i]], B[j, :lb[j]])
    return out

def pack(seqs):
    L = np.array([len(s) for s in seqs], np.int64)
    A = np.zeros((len(seqs), L.max(), seqs[0].shape[1]), np.float32)
    for i, s in enumerate(seqs): A[i, :len(s)] = s
    return A, L

KEEP = list(range(11, 17)) + list(range(17, 59))

def build(win, order, B=4):
    X, meta = [], []
    for s, x in raw.items():
        xc = clean2(x, win, order, B)
        if xc is None: continue
        X.append(xc[:, KEEP, :2].reshape(len(xc), -1).astype(np.float32))
        meta.append((s, info.loc[s, "Gloss"], info.loc[s, "split"]))
    return X, pd.DataFrame(meta, columns=["stem", "gloss", "split"])

def evaluate(X, meta):
    tr = np.where(meta.split == "train")[0]
    A_tr, l_tr = pack([X[i] for i in tr])
    gl_tr = meta.gloss.values[tr]
    classes = sorted(set(gl_tr))
    out = {}
    for sp in ["val", "test"]:
        q = np.where(meta.split == sp)[0]
        q = np.array([i for i in q if meta.gloss.values[i] in classes])
        A_q, l_q = pack([X[i] for i in q])
        Dm = dtw_matrix(A_q, l_q, A_tr, l_tr)
        M = np.stack([Dm[:, gl_tr == c].min(1) for c in classes], axis=1)
        true = np.array([classes.index(g) for g in meta.gloss.values[q]])
        rank = (M < M[np.arange(len(q)), true][:, None]).sum(1)
        out[sp] = dict(n=len(q), top1=(rank == 0).mean(), top5=(rank < 5).mean())
    return out, len(classes)

rows = []
for name, win, order in [("بلا تنعيم", 1, 2), ("SG 5/2", 5, 2), ("SG 7/2", 7, 2)]:
    t0 = time.time()
    X, meta = build(win, order)
    res, ncls = evaluate(X, meta)
    row = dict(variant=name)
    for sp in ["val", "test"]:
        p, n = res[sp]["top1"], res[sp]["n"]
        row[f"{sp}_top1"] = f"{p:.1%} ±{np.sqrt(p * (1 - p) / n):.1%}"
        row[f"{sp}_top5"] = f"{res[sp]['top5']:.1%}"
        row[f"{sp}_n"] = n
    rows.append(row)
    print(f"{name}: انتهى في {time.time() - t0:.0f}ث")

print(f"\nعدد الـ glosses: {ncls} | الصدفة: top1={1 / ncls:.1%} top5={5 / ncls:.1%}")
print(pd.DataFrame(rows).to_string(index=False))

مجلد الفيديوهات: /kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/videos
تم تحميل 1225 | بدون أبعاد (افترضنا 4:3): 0
قرارات: {'unknown': 22, 'right': 10, 'mixed': 10, 'flip': 4} | سيُعكس: ['P17', 'P49', 'P6', 'P9'] (القديم: P17 P49 P6 P9)
فيديوهات معكوسة: 174
train: 591 | طول اليد في القالب: 0.36 / 0.39
بلا تنعيم: انتهى في 21ث
SG 5/2: انتهى في 18ث
SG 7/2: انتهى في 17ث

عدد الـ glosses: 40 | الصدفة: top1=2.5% top5=12.5%
  variant    val_top1 val_top5  val_n   test_top1 test_top5  test_n
بلا تنعيم 62.4% ±4.0%    89.9%    149 57.3% ±2.3%     80.8%     475
   SG 5/2 63.1% ±4.0%    92.6%    149 58.7% ±2.3%     81.5%     475
   SG 7/2 61.1% ±4.0%    90.6%    149 57.3% ±2.3%     81.5%     475


In [8]:
import shutil, os
W = "/kaggle/working"
os.makedirs(f"{W}/extra", exist_ok=True)
for f in ["participant_decisions.csv", "manifest.csv"]:
    shutil.copy(f"{W}/{f}", f"{W}/extra/{f}")
shutil.copytree(f"{W}/keypoints_canon", f"{W}/extra/keypoints_canon", dirs_exist_ok=True)
shutil.make_archive(f"{W}/canon_backup", "zip", f"{W}/extra")
print(f"canon_backup.zip: {os.path.getsize(f'{W}/canon_backup.zip')/1e6:.0f} MB")

canon_backup.zip: 17 MB


In [9]:
import pandas as pd, glob, re

# 1) ASL Citizen
ASL_ROOT = glob.glob("/kaggle/input/**/ASL_Citizen", recursive=True)[0]
df = pd.concat([pd.read_csv(f"{ASL_ROOT}/splits/{s}.csv") for s in ["train", "val", "test"]])
vocab = sorted(df["Gloss"].unique())
print("عدد glosses في الداتا سيت:", len(vocab))          # يفترض ≈ 2731
print("عيّنة:", vocab[:15])

norm = lambda g: re.sub(r"[^A-Z0-9]", "", str(g).upper())   # DON'T-LIKE -> DONTLIKE
vocab_n = {norm(g) for g in vocab}

# 2) ملف الجمل (يُبحث عنه تلقائياً في input و working)
hits = (glob.glob("/kaggle/input/**/sentences_glosses.csv", recursive=True)
        + glob.glob("/kaggle/working/sentences_glosses.csv"))
assert hits, "لم أجد sentences_glosses.csv، تأكد من رفعه (Add Input) أو وضعه في /kaggle/working"
sg = pd.read_csv(hits[0])
print("الملف:", hits[0], "| عدد الجمل:", len(sg), "\n")

# 3) حساب التغطية
tot = found = fs = full = 0
missing_all = []
for _, r in sg.iterrows():
    toks = str(r["gloss"]).split()
    miss = []
    for t in toks:
        if t.startswith("fs-"):
            fs += 1
            continue                     # تهجئة بالأصابع: خارج القاموس أصلاً
        tot += 1
        if norm(t) in vocab_n:
            found += 1
        else:
            miss.append(t)
    full += (len(miss) == 0)
    missing_all += miss
    mark = "✅" if not miss else "❌"
    print(f"{mark} {r['gloss']:42} ناقص: {miss if miss else '-'}")

print(f"\nالكلمات الموجودة: {found}/{tot} = {found/tot:.1%}  (+ {fs} اسم بالتهجئة)")
print(f"جمل مغطاة بالكامل: {full}/{len(sg)} = {full/len(sg):.1%}")
print("الكلمات الناقصة:", sorted(set(missing_all)))


عدد glosses في الداتا سيت: 2731
عيّنة: ['1DOLLAR', '5DOLLARS', '8HOUR', '9OCLOCK', 'A-LINEBOB', 'ABBREVIATE', 'ABOUT1', 'ABOUT2', 'ABOVE', 'ABSOLUTELYNOTHING', 'ACCENT', 'ACCEPT', 'ACCESS', 'ACCIDENT', 'ACCOMPLISH']
الملف: /kaggle/input/datasets/danahaffar/glosses/sentences_glosses.csv | عدد الجمل: 20 

✅ ME APPLE LIKE                              ناقص: -
❌ ME PIZZA DON'T-LIKE                        ناقص: ['PIZZA']
✅ YOUR NAME WHAT                             ناقص: -
❌ YOU LIVE WHERE                             ناقص: ['LIVE']
❌ TOMORROW ME GO STORE                       ناقص: ['STORE']
❌ MY BROTHER LIVE fs-LONDON                  ناقص: ['LIVE']
✅ NEXT WEEK ME MY MOTHER VISIT               ناقص: -
❌ YESTERDAY YOU BREAKFAST EAT                ناقص: ['BREAKFAST', 'EAT']
❌ ME WANT PIZZA EAT                          ناقص: ['WANT', 'PIZZA', 'EAT']
✅ ME HAPPY NOT                               ناقص: -
❌ ME CAN'T SWIM                              ناقص: ["CAN'T", 'SWIM']
✅ ME MUST GO HOME       

In [10]:
import re
from collections import defaultdict

# الاسم الأساسي بدون اللاحقة الرقمية: EAT1 -> EAT
base = lambda g: re.sub(r"\d+$", "", norm(g))
variants = defaultdict(list)
for g in vocab:
    variants[base(g)].append(g)

# ---------- (أ) ابحث عن الكلمات الناقصة وبدائلها ----------
probe = ["EAT", "FOOD", "WANT", "SHE", "HE", "LIVE", "RESIDE", "STORE", "SHOP",
         "MARKET", "SWIM", "BREAKFAST", "PIZZA", "CAN", "CANNOT", "GOOD",
         "MORNING", "HOW"]
print("=== بحث عن الناقص وبدائله ===")
for p in probe:
    hits = [g for g in vocab if base(g) == p or norm(g).startswith(p)]
    print(f"{p:10} → {hits[:10]}{' ...' if len(hits) > 10 else ''}")

# ---------- (ب) التغطية بعد مطابقة الاسم الأساسي ----------
tot = found = full = 0
still_missing = []
print("\n=== التغطية بعد تجاهل اللواحق الرقمية ===")
for _, r in sg.iterrows():
    miss = []
    for t in str(r["gloss"]).split():
        if t.startswith("fs-"):
            continue
        tot += 1
        if base(t) in variants:
            found += 1
        else:
            miss.append(t)
    full += (not miss)
    still_missing += miss
    print(f"{'✅' if not miss else '❌'} {r['gloss']:42} ناقص: {miss if miss else '-'}")

print(f"\nالكلمات الموجودة: {found}/{tot} = {found/tot:.1%}")
print(f"جمل مغطاة بالكامل: {full}/{len(sg)} = {full/len(sg):.1%}")
print("الناقص فعلاً:", sorted(set(still_missing)))

=== بحث عن الناقص وبدائله ===
EAT        → ['EAT1', 'EAT2']
FOOD       → []
WANT       → ['WANT1', 'WANT2']
SHE        → ['SHEEP', 'SHELF']
HE         → ['HE', 'HEAD', 'HEADACHE', 'HEADBACK', 'HEADCOLD', 'HEADGRIND', 'HEADPHONES', 'HEALTH', 'HEAP', 'HEAR1'] ...
LIVE       → ['LIVE1', 'LIVE2']
RESIDE     → []
STORE      → []
SHOP       → ['SHOP1', 'SHOP2']
MARKET     → []
SWIM       → ['SWIM1', 'SWIM2']
BREAKFAST  → ['BREAKFAST1', 'BREAKFAST2']
PIZZA      → ['PIZZA1', 'PIZZA2', 'PIZZACUTTER']
CAN        → ['CAN1', 'CAN2', 'CANADA', 'CANCEL1', 'CANCEL2', 'CANCELLATION', 'CANCER1', 'CANCER2', 'CANDLE', 'CANDY1'] ...
CANNOT     → ['CANNOT']
GOOD       → ['GOOD', 'GOODYGOODY', 'GOODYGOODYSHOEOPPOSITE']
MORNING    → ['MORNING']
HOW        → ['HOW1', 'HOW2', 'HOWL']

=== التغطية بعد تجاهل اللواحق الرقمية ===
✅ ME APPLE LIKE                              ناقص: -
✅ ME PIZZA DON'T-LIKE                        ناقص: -
✅ YOUR NAME WHAT                             ناقص: -
✅ YOU LIVE WHERE            

In [11]:
# كل الـ glosses الوظيفية التي قد يخرجها محرّكك
NEED = """ME MY MYSELF YOU YOUR HE HIS WE OUR THEY THEIR IT ITS
MUST SHOULD MAYBE CAN CANNOT NOT FUTURE BEFORE FINISH
TOMORROW YESTERDAY TODAY NOW TONIGHT LATER MORNING AFTERNOON EVENING NIGHT
MONDAY TUESDAY WEDNESDAY THURSDAY FRIDAY SATURDAY SUNDAY
WEEK MONTH YEAR DAY HOUR MINUTE NEXT LAST THIS EVERY THAT THESE THOSE NO ALL SOME EACH
WHAT WHERE WHEN WHO WHY HOW WHICH IF UNLESS BECAUSE BUT OR SO ALTHOUGH
HELLO GOODBYE OK THANK-YOU EXCUSE-ME PLEASE SORRY YES
MOTHER FATHER CHILD PEOPLE ONE TWO THREE TEN HUNDRED""".split()

ok_, miss_ = [], []
for w in NEED:
    (ok_ if base(w) in variants else miss_).append(w)
print("موجودة :", len(ok_), "| ناقصة:", len(miss_))
print("الناقصة:", miss_)
print("\nتنويعات بعض الموجودة:")
for w in ["CAN", "HOW", "BEFORE", "FUTURE", "NEXT", "LAST", "THANK-YOU", "HIS", "MUST", "ME", "YOU"]:
    print(f"  {w:10} → {variants.get(base(w), 'ناقصة')}")

# ---------- جدول التحويل المبدئي ----------
ALIASES = {
    "SHE": ["HE"],
    "STORE": ["SHOP"],
    "CAN'T": ["CANNOT"],
    "GOOD-MORNING": ["GOOD", "MORNING"],
    "HOW-ARE-YOU": ["HOW", "YOU"],
}

def expand(gloss):
    out = []
    for t in str(gloss).split():
        out += ALIASES.get(t, [t])
    return out

tot = found = full = 0
for _, r in sg.iterrows():
    toks = [t for t in expand(r["gloss"]) if not t.startswith("fs-")]
    miss = [t for t in toks if base(t) not in variants]
    tot += len(toks); found += len(toks) - len(miss); full += (not miss)
    if miss:
        print("❌", r["gloss"], "→", miss)

print(f"\nبعد جدول التحويل: {found}/{tot} = {found/tot:.1%} | جمل كاملة: {full}/{len(sg)} = {full/len(sg):.1%}")

موجودة : 74 | ناقصة: 13
الناقصة: ['THEIR', 'IT', 'ITS', 'TONIGHT', 'EVENING', 'THIS', 'EVERY', 'THOSE', 'UNLESS', 'ALTHOUGH', 'GOODBYE', 'EXCUSE-ME', 'TEN']

تنويعات بعض الموجودة:
  CAN        → ['CAN1', 'CAN2']
  HOW        → ['HOW1', 'HOW2']
  BEFORE     → ['BEFORE']
  FUTURE     → ['FUTURE']
  NEXT       → ['NEXT']
  LAST       → ['LAST']
  THANK-YOU  → ['THANKYOU']
  HIS        → ['HIS']
  MUST       → ['MUST']
  ME         → ['ME']
  YOU        → ['YOU']

بعد جدول التحويل: 61/61 = 100.0% | جمل كاملة: 20/20 = 100.0%


In [12]:
probe = ["BYE", "GOODBYE", "FAREWELL", "EXCUSE", "SORRY", "PARDON", "10", "TEN", "NUMBER"]
print("=== بحث ===")
for p in probe:
    hits = [g for g in vocab if base(g) == p or norm(g).startswith(p)]
    print(f"{p:10} → {hits[:10]}")

print("\n=== أسماء تبدأ برقم ===")
print([g for g in vocab if g[0].isdigit()][:40])

print("\n=== الأرقام بالحروف ===")
for w in ["ZERO","ONE","TWO","THREE","FOUR","FIVE","SIX","SEVEN","EIGHT","NINE","TEN",
          "ELEVEN","TWELVE","TWENTY","HUNDRED","THOUSAND","MILLION"]:
    print(f"{w:9} → {variants.get(base(w), 'ناقصة')}")

=== بحث ===
BYE        → ['BYE']
GOODBYE    → []
FAREWELL   → []
EXCUSE     → ['EXCUSE']
SORRY      → ['SORRY']
PARDON     → []
10         → []
TEN        → ['TENDENCY', 'TENDYOUR', 'TENNIS', 'TENT']
NUMBER     → ['NUMBERS']

=== أسماء تبدأ برقم ===
['1DOLLAR', '5DOLLARS', '8HOUR', '9OCLOCK']

=== الأرقام بالحروف ===
ZERO      → ['ZERO']
ONE       → ['ONE']
TWO       → ['TWO']
THREE     → ['THREE']
FOUR      → ['FOUR']
FIVE      → ناقصة
SIX       → ['SIX']
SEVEN     → ['SEVEN']
EIGHT     → ['EIGHT']
NINE      → ['NINE']
TEN       → ناقصة
ELEVEN    → ['ELEVEN']
TWELVE    → ناقصة
TWENTY    → ناقصة
HUNDRED   → ['HUNDRED']
THOUSAND  → ناقصة
MILLION   → ناقصة


In [13]:
import string
print("حروف مفردة:", [c for c in string.ascii_uppercase if c in set(vocab)] or "لا يوجد")

print("\n=== النفي المدمج ===")
for w in ["DON'T-LIKE", "DON'T-WANT", "DON'T-KNOW", "DON'T-UNDERSTAND", "DON'T-NEED",
          "MUST-NOT", "SHOULD-NOT", "SHOULD", "MAYBE", "NOT", "FINISH", "PLEASE"]:
    print(f"{w:18} → {variants.get(base(w), 'ناقصة')}")

حروف مفردة: لا يوجد

=== النفي المدمج ===
DON'T-LIKE         → ['DONTLIKE']
DON'T-WANT         → ['DONTWANT']
DON'T-KNOW         → ['DONTKNOW']
DON'T-UNDERSTAND   → ناقصة
DON'T-NEED         → ['DONTNEED']
MUST-NOT           → ناقصة
SHOULD-NOT         → ناقصة
SHOULD             → ['SHOULD']
MAYBE              → ['MAYBE']
NOT                → ['NOT']
FINISH             → ['FINISH']
PLEASE             → ['PLEASE']


In [14]:
import glob, re, pandas as pd
ASL_ROOT = glob.glob("/kaggle/input/**/ASL_Citizen", recursive=True)[0]
df = pd.concat([pd.read_csv(f"{ASL_ROOT}/splits/{s}.csv") for s in ["train", "val", "test"]])
vocab = sorted(df["Gloss"].unique())


In [15]:
for k in ["FIVE", "TEN", "TWELVE", "TWENTY", "THOUSAND", "MILLION", "DOLLAR", "UNDERSTAND", "LETTER", "ALPHABET", "SPELL"]:
    print(f"{k:11} →", [g for g in vocab if k in g][:12])

FIVE        → []
TEN         → ['ATTENTION', 'CALLATTENTION', 'INTENTIONAL', 'KINDERGARTEN', 'KITTEN', 'LISTEN', 'OFTEN', 'PERSISTENT', 'PRETEND', 'SENTENCE1', 'SENTENCE2', 'TENDENCY']
TWELVE      → []
TWENTY      → []
THOUSAND    → []
MILLION     → []
DOLLAR      → ['1DOLLAR', '5DOLLARS']
UNDERSTAND  → ['MISUNDERSTAND', 'NOTUNDERSTAND', 'UNDERSTAND']
LETTER      → ['LETTER1', 'LETTER2']
ALPHABET    → ['ALPHABET']
SPELL       → ['FINGERSPELL', 'FINGERSPELLING']


In [16]:
import cv2, numpy as np, pandas as pd
print(f"فيديوهات: {len(df):,} | glosses: {df['Gloss'].nunique():,} | مصوّرين: {df['Participant ID'].nunique()}")
print("التوزيع:\n", df.groupby("split").size().to_dict() if "split" in df else "")

per_gloss = df.groupby("Gloss").size()
print("\nفيديوهات لكل gloss:", per_gloss.describe()[["min", "25%", "50%", "75%", "max"]].to_dict())
print("أقل 5:", per_gloss.nsmallest(5).to_dict())

per_signer = df.groupby("Participant ID").size()
print("\nفيديوهات لكل مصوّر:", per_signer.describe()[["min", "50%", "max"]].to_dict())

# خصائص فيديو من عيّنة عشوائية
sample = df.sample(150, random_state=0)
rows = []
for f in sample["Video file"]:
    cap = cv2.VideoCapture(f"{ASL_ROOT}/videos/{f}")
    n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
    rows.append(dict(w=cap.get(3), h=cap.get(4), fps=fps, dur=n / fps if fps else np.nan))
    cap.release()
v = pd.DataFrame(rows)
print("\nالدقة الأشيع:", v.groupby(["w", "h"]).size().sort_values(ascending=False).head(4).to_dict())
print("fps:", v["fps"].round().value_counts().head(3).to_dict())
print("المدة (ث):", v["dur"].describe()[["min", "50%", "max"]].round(1).to_dict())

فيديوهات: 83,399 | glosses: 2,731 | مصوّرين: 52
التوزيع:
 

فيديوهات لكل gloss: {'min': 21.0, '25%': 30.0, '50%': 31.0, '75%': 31.0, 'max': 45.0}
أقل 5: {'BEE2': 21, 'LETTUCE2': 21, 'BELT2': 22, 'CLOUD2': 22, 'DEMAND2': 22}

فيديوهات لكل مصوّر: {'min': 2.0, '50%': 1496.0, 'max': 3004.0}

الدقة الأشيع: {(640.0, 480.0): 148, (960.0, 540.0): 2}
fps: {30.0: 130, 31.0: 10, 25.0: 4}
المدة (ث): {'min': 1.0, '50%': 2.7, 'max': 8.0}


In [17]:
import os, pandas as pd

sp = pd.concat([pd.read_csv(f"{ASL_ROOT}/splits/{s}.csv").assign(split=s)
                for s in ["train", "val", "test"]])
print("التوزيع:", sp.groupby("split").size().to_dict(),
      "| مصوّرون لكل split:", sp.groupby("split")["Participant ID"].nunique().to_dict())

sp["b"] = sp["Gloss"].map(base)
cov = sp.groupby("Participant ID").agg(videos=("Gloss", "size"), glosses=("Gloss", "nunique"))
cov["قاموس%"] = (cov.glosses / sp["Gloss"].nunique() * 100).round(1)

if "NEED" in globals():
    need_b = {base(w) for w in NEED}
    cov["كلماتنا"] = (sp[sp.b.isin(need_b)].groupby("Participant ID")["b"].nunique()
                      .reindex(cov.index).fillna(0).astype(int))
    print(f"كلماتنا الوظيفية الموجودة في القاموس: {len(need_b & set(sp.b))}")

dec = "/kaggle/working/participant_decisions.csv"
if os.path.exists(dec):
    cov["اليد"] = pd.read_csv(dec, index_col=0)["decision"].reindex(cov.index).fillna("غير محسوب")
else:
    print("ملف participant_decisions.csv غير موجود، عمود اليد محذوف")

print(cov.sort_values("glosses", ascending=False).head(12))

التوزيع: {'test': 32941, 'train': 40154, 'val': 10304} | مصوّرون لكل split: {'test': 11, 'train': 35, 'val': 6}
كلماتنا الوظيفية الموجودة في القاموس: 74
                videos  glosses  قاموس%  كلماتنا     اليد
Participant ID                                           
P48               2999     2731   100.0       74    mixed
P49               3000     2730   100.0       74     flip
P18               2998     2729    99.9       74    right
P52               2728     2728    99.9       74    right
P21               2997     2727    99.9       74    mixed
P33               3000     2724    99.7       74    mixed
P39               3000     2724    99.7       74  unknown
P22               2997     2723    99.7       74    mixed
P17               3000     2722    99.7       74     flip
P11               3000     2719    99.6       73    right
P9                2988     2715    99.4       73     flip
P26               2999     2712    99.3       74    right


In [18]:
# بناء بنك الاشارات : 
#اختيار الفيديو لكل غلوس :
import os, re, glob, shutil, subprocess, cv2, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor

BANK = "/kaggle/working/sign_bank"
os.makedirs(f"{BANK}/clips", exist_ok=True)

PRIORITY    = ["P18", "P48", "P52"]     # المرجعي ثم الاحتياطيان
FLIP        = set()                     # المصوّرون الذين يُعكسون أفقياً (لا أحد منهم الآن)
ONLY_WORDS  = None                      # مثال: list(NEED) لبنك مصغّر

sp = pd.concat([pd.read_csv(f"{ASL_ROOT}/splits/{s}.csv") for s in ["train", "val", "test"]])
sp["b"] = sp["Gloss"].map(base)
if ONLY_WORDS:
    sp = sp[sp["b"].isin({base(w) for w in ONLY_WORDS})]

rank = {p: i for i, p in enumerate(PRIORITY)}
sp["rank"] = sp["Participant ID"].map(rank).fillna(99)
pick = (sp.sort_values(["Gloss", "rank", "Video file"])
          .drop_duplicates("Gloss")
          .rename(columns={"Participant ID": "signer"}).reset_index(drop=True))
pick["key"] = pick["Gloss"].map(norm)

print("glosses في البنك:", len(pick))
print("من أي مصوّر:", pick["signer"].value_counts().head(5).to_dict())

glosses في البنك: 2731
من أي مصوّر: {'P18': 2729, 'P48': 2}


In [19]:
# قص الصمت واعادة الترميز 
def active_range(path, pad=4):
    """يحدد أول وآخر لحظة فيها حركة، بحساب الفرق بين الإطارات."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    prev, d = None, []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        g = cv2.GaussianBlur(cv2.cvtColor(cv2.resize(fr, (160, 120)),
                                          cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)
        d.append(0.0 if prev is None else float(np.abs(g - prev).mean()))
        prev = g
    cap.release()
    n = len(d)
    if n < 5:
        return 0.0, None, n / fps
    d = np.convolve(d, np.ones(5) / 5, mode="same")
    thr = max(0.8, 0.25 * np.percentile(d, 90))
    idx = np.where(d > thr)[0]
    if len(idx) == 0:
        return 0.0, None, n / fps
    s, e = max(0, idx[0] - pad), min(n - 1, idx[-1] + pad)
    return s / fps, (e + 1) / fps, n / fps

def build_clip(row):
    src = f"{ASL_ROOT}/videos/{row['Video file']}"
    dst = f"{BANK}/clips/{row['key']}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 2000:
        return row["key"], dst, None, None
    s, e, full = active_range(src)
    vf = "scale=480:360,fps=25" + (",hflip" if row["signer"] in FLIP else "")
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ss", f"{s:.3f}"]
    if e is not None:
        cmd += ["-to", f"{e:.3f}"]
    cmd += ["-vf", vf, "-an", "-c:v", "libx264", "-crf", "26",
            "-preset", "veryfast", "-pix_fmt", "yuv420p", dst]
    r = subprocess.run(cmd, capture_output=True, text=True)
    ok = r.returncode == 0 and os.path.exists(dst) and os.path.getsize(dst) > 2000
    return row["key"], (dst if ok else None), full, ((e or full) - s)

# جرّب أولاً على 20 فقط
test = [build_clip(r) for _, r in pick.head(20).iterrows()]
print("الأصلي → بعد القص (ث):", [(round(f, 1), round(t, 1)) for _, _, f, t in test if f])

الأصلي → بعد القص (ث): [(3.2, np.float64(2.0)), (2.9, np.float64(2.0)), (2.8, np.float64(2.4)), (2.7, np.float64(2.1)), (3.1, np.float64(2.2)), (2.6, np.float64(1.9)), (3.2, np.float64(2.0)), (3.1, np.float64(2.1)), (2.7, np.float64(2.1)), (2.7, np.float64(2.0)), (3.2, np.float64(2.1)), (2.6, np.float64(1.8)), (3.0, np.float64(2.2)), (2.5, np.float64(1.5)), (2.9, np.float64(2.1)), (3.0, np.float64(2.5)), (3.3, np.float64(2.8)), (3.1, np.float64(2.1)), (2.9, np.float64(2.2)), (2.8, np.float64(2.1))]


In [20]:
# فحص بصري 
from IPython.display import Video, display
for k in [t[0] for t in test[:4]]:
    print(k)
    display(Video(f"{BANK}/clips/{k}.mp4", embed=True, width=320))

1DOLLAR


5DOLLARS


8HOUR


9OCLOCK


In [21]:
import time
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    res200 = list(ex.map(build_clip, [r for _, r in pick.iloc[20:220].iterrows()]))
dt = time.time() - t0
print(f"200 مقطع في {dt:.0f}ث → الكل تقريباً {dt/200*len(pick)/60:.0f} دقيقة")

200 مقطع في 82ث → الكل تقريباً 19 دقيقة


In [22]:
import time
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    res = list(ex.map(build_clip, [r for _, r in pick.iterrows()]))

pick["file"] = [r[1] for r in res]
fails = pick[pick["file"].isna()]
print(f"نجح: {pick['file'].notna().sum()} | فشل: {len(fails)} | الوقت: {(time.time()-t0)/60:.1f} دقيقة")
if len(fails):
    print("أمثلة الفاشل:", fails["Gloss"].head(10).tolist())

نجح: 2731 | فشل: 0 | الوقت: 16.8 دقيقة


In [23]:
ok = pick[pick["file"].notna()].copy()
exact = dict(zip(ok["key"], ok["file"]))

# المتغيرات: EAT1 و EAT2 تُجمع تحت الاسم الأساسي EAT، ويُختار الأقل رقماً
VARIANT_PICK = {}            # مثال: {"CAN": "CAN2"} بعد مراجعة الفيديو بالعين
by_base = {}
for g in ok["Gloss"]:
    by_base.setdefault(base(g), []).append(g)

lookup = dict(exact)
for b, gs in by_base.items():
    if b not in lookup:
        chosen = VARIANT_PICK.get(b) or sorted(gs, key=lambda x: (len(x), x))[0]
        lookup[b] = exact[norm(chosen)]

pd.DataFrame({"key": list(lookup), "file": [os.path.basename(f) for f in lookup.values()]}
             ).to_csv(f"{BANK}/lookup.csv", index=False)
ok[["Gloss", "key", "signer", "Video file", "file"]].to_csv(f"{BANK}/bank_manifest.csv", index=False)

shutil.make_archive("/kaggle/working/sign_bank", "zip", BANK)
mb = os.path.getsize("/kaggle/working/sign_bank.zip") / 1e6
print(f"مفاتيح البحث: {len(lookup)} | حجم البنك: {mb:.0f} MB")

multi = {b: gs for b, gs in by_base.items() if len(gs) > 1 and b in {base(w) for w in NEED}}
print("\nكلماتنا ذات التنويعات (راجعها بالعين):")
for b, gs in sorted(multi.items()):
    print(" ", b, "→", gs)

مفاتيح البحث: 3040 | حجم البنك: 139 MB

كلماتنا ذات التنويعات (راجعها بالعين):
  CAN → ['CAN1', 'CAN2']
  HOW → ['HOW1', 'HOW2']
  NIGHT → ['NIGHT1', 'NIGHT2']
  THEY → ['THEY1', 'THEY2']
  WHAT → ['W.H.A.T', 'WHAT1', 'WHAT2']


In [24]:
from IPython.display import Video
Video(f"{BANK}/clips/EAT1.mp4", embed=True, width=320)

In [25]:
from IPython.display import Video, display

# مقاطع التنويعات: WHAT.mp4 هنا هو مقطع W.H.A.T
for k in ["CAN1", "CAN2", "HOW1", "HOW2", "NIGHT1", "NIGHT2", "THEY1", "THEY2", "WHAT1", "WHAT2", "WHAT"]:
    print(k)
    display(Video(f"{BANK}/clips/{k}.mp4", embed=True, width=240))

# هل توجد حالات مشابهة أخرى؟ (اسم يحوي رموزاً ويطابق كلمة من قاموسنا بعد التطبيع)
need_b = {base(w) for w in NEED} | {norm(w) for w in NEED}
print(ok[(ok["Gloss"] != ok["key"]) & ok["key"].isin(need_b)][["Gloss", "key"]])

CAN1


CAN2


HOW1


HOW2


NIGHT1


...

THEY2


WHAT1


WHAT2


WHAT


        Gloss   key
2589  W.H.A.T  WHAT


In [26]:
VARIANT_PICK = {"WHAT": "WHAT1", "CAN": "CAN1", "HOW": "HOW1",
                "NIGHT": "NIGHT1", "THEY": "THEY1"}

for b, g in VARIANT_PICK.items():
    lookup[b] = exact[norm(g)]            # يُلغي حتى المفتاح الدقيق WHAT

pd.DataFrame({"key": list(lookup), "file": [os.path.basename(f) for f in lookup.values()]}
             ).to_csv(f"{BANK}/lookup.csv", index=False)
shutil.make_archive("/kaggle/working/sign_bank", "zip", BANK)
print("حجم البنك:", round(os.path.getsize("/kaggle/working/sign_bank.zip") / 1e6), "MB")

from IPython.display import FileLink
FileLink("/kaggle/working/sign_bank.zip")

حجم البنك: 139 MB


/kaggle/working/sign_bank.zip

In [27]:
import os, glob
z = "/kaggle/working/sign_bank.zip"
print("الـ zip موجود؟", os.path.exists(z), "| الحجم:", round(os.path.getsize(z)/1e6) if os.path.exists(z) else "-", "MB")
print("عدد المقاطع:", len(glob.glob("/kaggle/working/sign_bank/clips/*.mp4")))
print("lookup.csv:", os.path.exists("/kaggle/working/sign_bank/lookup.csv"))

الـ zip موجود؟ True | الحجم: 139 MB
عدد المقاطع: 2731
lookup.csv: True


In [28]:
from IPython.display import FileLink
FileLink("/kaggle/working/sign_bank.zip")

/kaggle/working/sign_bank.zip

In [ ]:
import os
from IPython.display import FileLink, display

PARTS = "/kaggle/working/parts"
os.makedirs(PARTS, exist_ok=True)
with open("/kaggle/working/sign_bank.zip", "rb") as f:
    i = 0
    while chunk := f.read(20 * 1024 * 1024):
        with open(f"{PARTS}/sign_bank.zip.part{i:02d}", "wb") as o:
            o.write(chunk)
        i += 1
print("عدد الأجزاء:", i)
for p in sorted(os.listdir(PARTS)):
    display(FileLink(f"{PARTS}/{p}"))

In [2]:
import os
z = "/kaggle/working/sign_bank.zip"
print("موجود:", os.path.exists(z), "| الأجزاء:", len(os.listdir("/kaggle/working/parts")) if os.path.exists("/kaggle/working/parts") else 0)

موجود: False | الأجزاء: 0


In [4]:
import os
from IPython.display import FileLink
os.chdir("/kaggle/working")
FileLink("sign_bank.zip")

/kaggle/working/sign_bank.zip